# I2 · Frozen DINOv2 ViT-L/14 at336px · fixed S/B/L blend
One new larger-backbone candidate after the user-reported0.97755winning F0+B336blend. Preserve all current checkpoints and exact winningCSV predictions. New model: official pretrained dinov2_vitl14,1024-D CLS features, P3 and fresh E4input1024→128→BN→ReLU→Dropout.3→20, augmentationseed123/headseed2026. Fixed prediction=(2/3)currentwinner+(1/3)L336; equal S/B/L backbone-branch weights. No TTA, weightsearch, extra labels, pseudo-labels or automatic Kaggle submission. A bigger model may lose; no score is promised.

## 1. Environment and source paths
Select T4 GPU. Keep compatible installed torch/torchvision. LARGE_BATCH_SIZE=2; reduce it to1 if memory is insufficient. CONTROL_BATCH_SIZE=8 stays separate to replay current sources. Models are loaded sequentially, with features stored on CPU.

In [ ]:
%pip -q install scikit-learn pandas pillow kagglehub


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
STORAGE_ROOT=Path('/content/drive/MyDrive/Car-Classification-Benchmark')
# Confirmed paths from your successful runs; edit only if moved.
F0_RUN_DIR=STORAGE_ROOT/'student_outputs_F0_Ensemble/20261108T091250Z_4ed04fc8'
I1_RUN_DIR=STORAGE_ROOT/'student_outputs_I1_ViTB336/1402790cb5a0'
LARGE_BATCH_SIZE=2
CONTROL_BATCH_SIZE=8
FORCE_REEXTRACT=False
DOWNLOAD_FROM_KAGGLE=True
DATA_ROOT=Path('/content/car_data')
for p in [F0_RUN_DIR/'ensemble_summary.json',I1_RUN_DIR/'summary.json',I1_RUN_DIR/'F0_I1_336_equal_branch_submission.csv']:
    if not p.is_file():raise FileNotFoundError(f'Missing preserved source artifact: {p}. Restore/set the exact source path; no source models will be retrained.')
if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN']=userdata.get('KAGGLE_API_TOKEN')
    DATA_ROOT=Path(kagglehub.competition_download('idnn-swe-3052-assignment-3-flowers-classification'))


## 2. Self-contained implementation
Verify currentsourcecheckpointSHA, classmaps, image hashes, recorded validation metrics and every winningCSV label before the L experiment. Official pretrained Lsource is pinned to the same upstreamSHA; its actual weight fingerprint becomes part of its separate cache/checkpoint. No B768features or normalization are used for L1024. The pinnedGit fallback handles transientGitHub502/503/504 without changing the revision. Source backbone hashes must match their recorded originals.

In [ ]:
"""I2: frozen official ViT-L/14 at336; fixed equal S/B/L backbone blend."""
from pathlib import Path
import gc,hashlib,json,random,subprocess,uuid
from urllib.error import HTTPError
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader,TensorDataset
from torchvision import datasets,transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
import torchvision
from pathlib import Path

import argparse, gc, hashlib, json, uuid

import numpy as np

import pandas as pd

import torch

from torch import nn

from torch.utils.data import DataLoader

from torchvision import datasets, transforms

from torchvision.transforms import InterpolationMode

from PIL import Image

HUB_REF='7764ea0f912e53c92e82eb78a2a1631e92725fc8'

SEEDS=[42,123,2026]

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def weight_digest(model):
    h=hashlib.sha256()
    for name,value in sorted(model.state_dict().items()):
        h.update(name.encode());h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def build_head(dim):
    return nn.Sequential(nn.Linear(dim,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,20))

class Images(torch.utils.data.Dataset):
    def __init__(self,paths,transform):self.paths,self.transform=paths,transform
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as image:return self.transform(image.convert('RGB'))

def load_data(root):
    root=Path(root)
    train,val=[datasets.ImageFolder(root/name) for name in ('train','val')]
    assert len(train)==700 and len(val)==200 and len(train.classes)==20
    assert train.class_to_idx==val.class_to_idx
    assert all(name.startswith(f'{i:03d}_') for i,name in enumerate(train.classes))
    sample=pd.read_csv(root/'sample_submission.csv',dtype={'image_id':str})
    assert {'image_id','label'}<=set(sample.columns) and sample.image_id.is_unique and sample.image_id.notna().all()
    files=sorted(p for p in (root/'test').iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})
    by_id={p.stem:p for p in files};ids=sample.image_id.tolist()
    assert len(by_id)==len(files) and set(by_id)==set(ids)
    return train.class_to_idx,[Path(p) for p,_ in val.samples],np.asarray(val.targets),[by_id[i] for i in ids],ids

def checked_checkpoint(path,model,dim,seed,mapping):
    p=torch.load(path,map_location='cpu',weights_only=True)
    assert p['backbone']==model and p['seed_aug']==123 and p['seed_head']==seed
    assert p['class_to_idx']==mapping
    for key in ('feature_mean','feature_std'):
        assert p[key].shape==(dim,) and torch.isfinite(p[key]).all()
    assert (p['feature_std']>0).all()
    expected={'hidden_dim':128,'use_bn':True,'dropout':.3}
    if dim==768:expected={'input_dim':768,**expected,'num_classes':20}
    assert p['head_config']==expected,'Checkpoint is not the requested E4 architecture'
    build_head(dim).load_state_dict(p['head_state_dict'],strict=True)
    return p

@torch.inference_mode()
def infer(backbone,checkpoints,paths,transform,device,batch_size=8):
    backbone.eval();assert not any(p.requires_grad for p in backbone.parameters())
    heads=[]
    for c in checkpoints:
        h=build_head(len(c['feature_mean'])).to(device)
        h.load_state_dict(c['head_state_dict'],strict=True);h.eval()
        heads.append((h,c['feature_mean'].to(device),c['feature_std'].to(device)))
    results=[[] for _ in heads]
    loader=DataLoader(Images(paths,transform),batch_size=batch_size,shuffle=False,num_workers=0)
    for index,images in enumerate(loader):
        features=backbone(images.to(device)).float()
        for j,(head,mean,std) in enumerate(heads):
            assert features.shape==(len(images),len(mean))
            p=head((features-mean)/std).softmax(1)
            assert p.shape==(len(images),20) and torch.isfinite(p).all()
            results[j].append(p.cpu())
        if index%40==0:print('PREDICT_BATCH',index+1,'/',len(loader),flush=True)
    return torch.stack([torch.cat(parts) for parts in results])

def metrics(target,probabilities):
    from sklearn.metrics import accuracy_score,f1_score
    pred=probabilities.argmax(1).numpy()
    return {'accuracy':float(accuracy_score(target,pred)),
        'macro_f1':float(f1_score(target,pred,labels=list(range(20)),average='macro',zero_division=0)),
        'correct':int((pred==target).sum()),'count':len(target)}

def reproduce(measured,expected,name):
    assert measured['correct']==expected['correct'] and abs(measured['accuracy']-expected['accuracy'])<1e-12, (
        f'{name} starting accuracy did not reproduce; diagnose checkpoint/source/preprocessing before exporting')

def export(path,ids,probabilities):
    pred=probabilities.argmax(1).numpy().astype(int)
    assert len(pred)==len(ids)==len(set(ids)) and ((pred>=0)&(pred<20)).all()
    pd.DataFrame({'image_id':list(ids),'label':pred}).to_csv(path,index=False,mode='x')
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)
SETTINGS={'epochs':35,'batch_size':64,'learning_rate':.001,'weight_decay':.001,'patience':7,'optimizer':'AdamW'}

SIZE=336

def transforms336():
    tail=[transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])]
    evaluation=transforms.Compose([transforms.Resize(384,interpolation=InterpolationMode.BICUBIC),transforms.CenterCrop(336),*tail])
    training=transforms.Compose([transforms.RandomResizedCrop(336,scale=(.8,1.),ratio=(.9,1.1),interpolation=InterpolationMode.BICUBIC),transforms.RandomHorizontalFlip(.5),*tail])
    return evaluation,training

def validate_source_data(base,data_root,mapping,ids):
    meta=base['feature_metadata']
    assert meta.get('backbone',meta.get('model'))=='dinov2_vitb14'
    assert base['hub_ref']==HUB_REF and meta['class_to_idx']==mapping and meta['test_ids']==ids
    assert meta['feature_dim']==768 and meta['seed_aug']==123 and meta['n_aug']==4 and meta['include_original'] is True
    root=Path(data_root).resolve()
    for split in meta['manifest']:
        for entry in (split['files'] if isinstance(split,dict) else split):
            path=(root/entry['path']).resolve()
            assert path.is_relative_to(root) and path.is_file() and sha(path)==entry['sha256'],'Data differs from original I0 run'

def load_official(name,expected,dim,device):
    try:
        model=torch.hub.load(f'facebookresearch/dinov2:{HUB_REF}',name,pretrained=True,trust_repo=True,skip_validation=True)
    except HTTPError as exc:
        if exc.code not in (502,503,504):raise
        root=Path(torch.hub.get_dir())/('dinov2_git_'+HUB_REF[:12])
        if not root.exists():
            subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/facebookresearch/dinov2.git',str(root)],check=True)
        subprocess.run(['git','-C',str(root),'checkout','--detach',HUB_REF],check=True)
        assert subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()==HUB_REF
        model=torch.hub.load(str(root),name,source='local',pretrained=True)
    model.requires_grad_(False).eval()
    assert model.embed_dim==dim
    if expected is not None:assert weight_digest(model)==expected,'Backbone does not match original source weights'
    return model.to(device)

def get_large_features(backbone,base,data_root,val_paths,target,test_paths,ids,mapping,cache_root,device,batch_size=8,force=False):
    train_ds=datasets.ImageFolder(Path(data_root)/'train')
    assert train_ds.class_to_idx==mapping and len(train_ds)==700
    train_paths=[Path(p) for p,_ in train_ds.samples]
    evaluation,augmentation=transforms336()
    assert not backbone.training and not any(p.requires_grad for p in backbone.parameters())
    fingerprint=weight_digest(backbone)
    assert backbone.embed_dim==1024
    meta={'pipeline':'I2-official-L336-v1','backbone':'dinov2_vitl14','hub_ref':HUB_REF,
        'weights_sha256':fingerprint,'image_size':336,'resize_size':384,'feature_dim':1024,
        'train_transform':repr(augmentation),'val_transform':repr(evaluation),'seed_aug':123,
        'n_aug':4,'include_original':True,'layout':'view-major','class_to_idx':mapping,
        'manifest':base['feature_metadata']['manifest'],'test_ids':ids,'batch_size':batch_size,
        'torch':str(torch.__version__),'torchvision':str(torchvision.__version__),'pillow':Image.__version__,'device':str(device)}
    signature=hashlib.sha256(json.dumps(meta,sort_keys=True).encode()).hexdigest()
    cache=Path(cache_root)/f'I2_dinov2_vitl14_336_p3_aug123_{signature}.pt';cache.parent.mkdir(parents=True,exist_ok=True)
    def valid(p):
        try:
            assert p['metadata']==meta
            for name,n in [('X_train',3500),('X_val',200),('X_test',len(ids))]:
                assert p[name].shape==(n,1024) and p[name].dtype==torch.float32 and torch.isfinite(p[name]).all()
            assert torch.equal(p['y_train'],torch.tensor(train_ds.targets).repeat(5))
            assert np.array_equal(p['y_val'].numpy(),target)
            return True
        except (KeyError,AssertionError,TypeError,AttributeError):return False
    if cache.exists() and not force:
        try:
            p=torch.load(cache,map_location='cpu',weights_only=True)
            if valid(p):print('CACHE_HIT',cache,flush=True);return p,signature,cache
            print('INVALID_CACHE; regenerating',cache,flush=True)
        except Exception as e:print('UNREADABLE_CACHE; regenerating',type(e).__name__,flush=True)
    set_seed(123)
    @torch.inference_mode()
    def extract(paths,transform,name):
        loader=DataLoader(Images(paths,transform),batch_size=batch_size,shuffle=False,num_workers=0)
        outputs=[]
        for index,batch in enumerate(loader):
            z=backbone(batch.to(device)).float().cpu()
            assert z.shape==(len(batch),1024) and torch.isfinite(z).all()
            outputs.append(z)
            if index%40==0:print('EXTRACT336',name,index+1,'/',len(loader),flush=True)
        return torch.cat(outputs)
    views=[extract(train_paths,evaluation,'original')]
    for index in range(4):views.append(extract(train_paths,augmentation,f'aug{index+1}'))
    p={'metadata':meta,'X_train':torch.cat(views),'X_val':extract(val_paths,evaluation,'validation'),
        'X_test':extract(test_paths,evaluation,'test'),'y_train':torch.tensor(train_ds.targets).repeat(5),'y_val':torch.tensor(target)}
    assert valid(p)
    temporary=cache.with_suffix('.'+uuid.uuid4().hex+'.tmp');torch.save(p,temporary);temporary.replace(cache)
    return p,signature,cache

def train_large(features,base,signature,output,device):
    assert base['training_config']==SETTINGS,'Source recipe differs from fixed I2 recipe'
    mean=features['X_train'].mean(0);std=features['X_train'].std(0,unbiased=False).clamp_min(1e-6)
    train=(features['X_train']-mean)/std;val=(features['X_val']-mean)/std
    set_seed(2026)
    head=build_head(1024).to(device)
    loader=DataLoader(TensorDataset(train,features['y_train']),batch_size=64,shuffle=True,num_workers=0,generator=torch.Generator().manual_seed(2026))
    optimizer=torch.optim.AdamW(head.parameters(),lr=.001,weight_decay=.001)
    best=-1.;bad=0;best_epoch=0;weights=None;history=[]
    for epoch in range(1,36):
        head.train();loss_sum=0.
        for x,y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss=torch.nn.functional.cross_entropy(head(x.to(device)),y.to(device));loss.backward();optimizer.step()
            loss_sum+=loss.item()*len(y)
        head.eval()
        with torch.inference_mode():p=head(val.to(device)).softmax(1).cpu()
        score=metrics(features['y_val'].numpy(),p)
        history.append({'epoch':epoch,'train_loss':loss_sum/len(train),**score})
        print('I2_HEAD_EPOCH',epoch,score,flush=True)
        if score['accuracy']>best:
            best,bad,best_epoch=score['accuracy'],0,epoch
            weights={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
        else:
            bad+=1
            if bad>=7:break
    head.load_state_dict(weights);head.eval()
    with torch.inference_mode():
        pv=head(val.to(device)).softmax(1).cpu()
        pt=head(((features['X_test']-mean)/std).to(device)).softmax(1).cpu()
    score=metrics(features['y_val'].numpy(),pv)
    path=Path(output)/'I2_dinov2_vitl14_336_p3_e4_aug123_head2026.pt'
    checkpoint={'backbone':'dinov2_vitl14','resolution':336,'hub_ref':HUB_REF,'seed_aug':123,'seed_head':2026,
        'head_state_dict':weights,'head_config':{'input_dim':1024,'hidden_dim':128,'use_bn':True,'dropout':.3,'num_classes':20},'class_to_idx':base['class_to_idx'],
        'feature_mean':mean,'feature_std':std,'feature_metadata':features['metadata'],'feature_signature':signature,
        'metrics':score,'best_accuracy':best,'best_epoch':best_epoch,'training_config':SETTINGS}
    with path.open('xb') as stream:torch.save(checkpoint,stream)
    pd.DataFrame(history).to_csv(Path(output)/'I2_training_history.csv',index=False)
    return pv,pt,path,checkpoint

def blend_large(current,large):
    assert current.shape==large.shape and current.ndim==2 and current.shape[1]==20
    for p in (current,large):
        assert torch.isfinite(p).all() and ((p>=0)&(p<=1)).all() and torch.allclose(p.sum(-1),torch.ones_like(p.sum(-1)),atol=1e-5)
    return (2/3)*current+(1/3)*large

def run_large(data_root,f0_run,i1_run,storage_root,large_batch_size=2,control_batch_size=8,force=False):
    assert large_batch_size>=1 and control_batch_size>=1
    mapping,val_paths,target,test_paths,ids=load_data(data_root)
    f0_run=Path(f0_run);i1_run=Path(i1_run)
    source=json.loads((f0_run/'ensemble_summary.json').read_text());reference=json.loads((i1_run/'summary.json').read_text())
    assert source['head_seeds']==SEEDS and source['hub_ref']==HUB_REF and source['augmentation_seed']==123
    assert np.allclose(source['weights'],[1/3]*3)
    rows={r['seed']:r for r in source['individual_validation']};assert set(rows)==set(SEEDS)
    old_paths=[f0_run/Path(rows[s]['checkpoint']).name for s in SEEDS]
    expected={Path(p).name:h for p,h in reference['source_checkpoint_sha256'].items()}
    for p in old_paths:assert sha(p)==expected[p.name],'F0 source differs from scored model'
    base_path=i1_run/Path(reference['new_checkpoint']).name
    assert sha(base_path)==reference['new_checkpoint_sha256'],'B336 source differs from scored model'
    old_paths.append(base_path);originals={str(p):sha(p) for p in old_paths}
    small=[checked_checkpoint(p,'dinov2_vits14',384,s,mapping) for p,s in zip(old_paths[:3],SEEDS)]
    base=checked_checkpoint(base_path,'dinov2_vitb14',768,2026,mapping)
    assert base['resolution']==336 and base['feature_metadata']['image_size']==336
    validate_source_data(base,data_root,mapping,ids)
    root=Path(storage_root)/'student_outputs_I2_ViTL336';output=root/uuid.uuid4().hex[:12];output.mkdir(parents=True,exist_ok=False)
    print('OUTPUT',output,flush=True)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    evaluation224=transforms.Compose([transforms.Resize(256,interpolation=InterpolationMode.BICUBIC),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    evaluation336,_=transforms336()
    backbone=load_official('dinov2_vits14',source['weights_sha256'],384,device)
    vs=infer(backbone,small,val_paths,evaluation224,device,control_batch_size)
    for index,s in enumerate(SEEDS):reproduce(metrics(target,vs[index]),rows[s],f'F0 seed{s}')
    reproduce(metrics(target,vs.mean(0)),source['ensemble_validation'],'F0 ensemble')
    ts=infer(backbone,small,test_paths,evaluation224,device,control_batch_size)
    del backbone;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    backbone=load_official('dinov2_vitb14',base['feature_metadata']['weights_sha256'],768,device)
    vb=infer(backbone,[base],val_paths,evaluation336,device,control_batch_size)[0]
    reproduce(metrics(target,vb),base['metrics'],'Preserved B336')
    tb=infer(backbone,[base],test_paths,evaluation336,device,control_batch_size)[0]
    current_val=.5*vs.mean(0)+.5*vb;current_test=.5*ts.mean(0)+.5*tb
    reproduce(metrics(target,current_val),reference['validation']['new_F0_B336_blend'],'Current0.97755blend')
    csv=i1_run/'F0_I1_336_equal_branch_submission.csv'
    old=pd.read_csv(csv,dtype={'image_id':str})
    assert old.image_id.tolist()==ids and np.array_equal(old.label.to_numpy(),current_test.argmax(1).numpy()),'Scored0.97755predictions changed'
    del backbone;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    backbone=load_official('dinov2_vitl14',None,1024,device)
    features,key,cache=get_large_features(backbone,base,data_root,val_paths,target,test_paths,ids,mapping,root/'cache_336',device,large_batch_size,force)
    pv,pt,path,checkpoint=train_large(features,base,key,output,device)
    measured={'current_best_S_B336':metrics(target,current_val),'new_L336_standalone':metrics(target,pv),
        'new_equal_S_B_L_blend':metrics(target,blend_large(current_val,pv))}
    for p in old_paths:assert sha(p)==originals[str(p)],'Original checkpoint modified'
    export(output/'CurrentBest_097755_control.csv',ids,current_test)
    export(output/'I2_ViTL336_standalone.csv',ids,pt)
    export(output/'F0_B336_L336_equal_backbone_submission.csv',ids,blend_large(current_test,pt))
    pd.DataFrame([{'model':name,**value} for name,value in measured.items()]).to_csv(output/'validation_comparison.csv',index=False)
    np.savez_compressed(output/'validation_probabilities.npz',current=current_val.numpy(),large=pv.numpy(),targets=target)
    result={'configuration':'(2/3)*current0.97755ensemble+(1/3)*L336; equal S/B/L backbone branches',
        'effective_head_weights':[1/9,1/9,1/9,1/3,1/3],'validation':measured,'new_checkpoint':str(path),'new_checkpoint_sha256':sha(path),
        'new_cache':str(cache),'cache_signature':key,'source_checkpoint_sha256':originals,'original_scored_predictions_reproduced':True,
        'reference_submission_sha256':sha(csv),'augmentation_seed':123,'head_seed':2026,'frozen_backbone':True,
        'TTA':False,'submitted':False,'test_labels_used':False,'prediction_count':len(ids),'historical_reference_Kaggle':.97755,
        'large_batch_size':large_batch_size,'control_batch_size':control_batch_size,
        'candidate_rule':'one pretrained architecture; fixedweights; export regardless of combined validation delta'}
    (output/'summary.json').write_text(json.dumps(result,indent=2))
    print('COMPLETE',output,json.dumps(result),flush=True)
    return output,result


## 3. Extract L336 features, train one new head and export
P3 uses one deterministic384→336crop plus four random geometry trainingviews at336, noColorJitter. Expect3500×1024train features from700images,200×1024validation, samplelength×1024test. Normalize only with newtrainingmean/populationSD, clamp1e-6. AdamWlr/decay.001,batch64,max35epochs,patience7; first strictly best validation accuracy, no F1tie-reset. Backbone stays frozen/eval/inference mode. No combined-validation improvement is required to export.

In [ ]:
output,summary=run_large(DATA_ROOT,F0_RUN_DIR,I1_RUN_DIR,STORAGE_ROOT,LARGE_BATCH_SIZE,CONTROL_BATCH_SIZE,FORCE_REEXTRACT)
from IPython.display import display
display(pd.DataFrame([{'model':name,**value} for name,value in summary['validation'].items()]))
print('PRIMARY NEW CANDIDATE:',output/'F0_B336_L336_equal_backbone_submission.csv')
print('StandaloneL336 diagnostic:',output/'I2_ViTL336_standalone.csv')
print('Verified0.97755control:',output/'CurrentBest_097755_control.csv')
print('Not submitted automatically; retain the0.97755submission.')


## 4. Keep and submit
Manually submit F0_B336_L336_equal_backbone_submission.csv as the one primary new candidate. It gives1/3weight to each backbone branch: each of the three F0Sheads contributes1/9, B336 contributes1/3 and L336 contributes1/3. Do not fit weights or choose subsets against the leaderboard. Preserve the winning0.97755CSV and all original checkpoints. Newuniqueoutputs/sharedcache under student_outputs_I2_ViTL336 include the Lhead, train-only statistics, rawfeatures, history, validation probabilities, comparison and provenance. Real Kaggle performance remains unknown until measured.